# Rendering Service

## Imports

In [12]:
import os
from pathlib import Path
from typing import *

from dotenv import load_dotenv


load_dotenv(dotenv_path=Path("../.env"))

True

In [ ]:
from python_utils.model_schemas.request import (
    ServiceConnectionDetails,
    get_service_connection_details,
    EndpointConfig,
)

rendering_service_conn_details: ServiceConnectionDetails = (
    get_service_connection_details(prefix="RENDERING")
)
print(f"{rendering_service_conn_details=}")

rendering_service_endpoint_config = EndpointConfig(
    connection=rendering_service_conn_details, endpoint="/render/asset"
)
render_asset_url: str = rendering_service_endpoint_config.build_url()
print(f"{render_asset_url=}")

rendering_service_conn_details=ServiceConnectionDetails(protocol='http', host='localhost', port=8000, timeout=30)
render_asset_url='http://localhost:8000//render/asset'


In [ ]:
from python_utils.model_schemas.common import Base64StringContent
from python_utils.model_schemas.files import (
    FileDetails,
    FileUpload,
    UploadFile,
    UploadFiles,
)
from python_utils.model_schemas.rendering import (
    RenderAssetRequest,
    RenderAssetResponse,
    RenderingDetails,
)
from python_utils.utils.file_utils import (
    encode_file_to_base64,
    get_usda_file_path,
    rglob_excluding,
)

asset_folder_path: Path = Path(
    "../sample_objects/Accessories/Decor/SM_Decor01"
).resolve()
assert (
    asset_folder_path.exists()
), f"No asset folder found at path: ({asset_folder_path=})"
assert (
    asset_folder_path.is_dir()
), f"Asset folder path is not a directory: ({asset_folder_path=})"

asset_usda_file_path: Path = [path for path in asset_folder_path.rglob("*.usda")].pop()
print(f"{asset_usda_file_path=}")
assert asset_usda_file_path.exists(), f"USDA file not found: {asset_usda_file_path}"

pattern: str = "*"
renderings_pattern: str = "_Capture-"
exclude_condition: Callable[[Path], bool] = lambda p: renderings_pattern in p.name
include_directories: bool = False
ignored_ancestors: str = str(Path("sample_objects/").resolve())
asset_file_paths: List[Path] = rglob_excluding(
    path=asset_folder_path,
    pattern=pattern,
    exclude_condition_met=exclude_condition,
    include_directories=include_directories,
    # ignored_ancestors=ignored_ancestors,
)
# Move the usda_file_path to the front of the list
usda_file_path: Path = get_usda_file_path(asset_file_paths)
print(f"{usda_file_path=}")
asset_file_paths.remove(usda_file_path)
asset_file_paths.insert(0, usda_file_path)
print(f"{len(asset_file_paths)=}")
print(f"{asset_file_paths=}")

asset_rendering_request: RenderAssetRequest = RenderAssetRequest(
    uid="abcde_12345",
    rendering_details=RenderingDetails(
        num_renders=int(os.environ.get("NUM_RENDERS", "6")),
        only_northern_hemisphere=(
            os.environ.get("ONLY_NORTHERN_HEMISPHERE", "true").lower() == "true"
        ),
        fast_mode=(os.environ.get("FAST_MODE", "false").lower() == "true"),
        render_alpha_maps=(
            os.environ.get("RENDER_ALPHA_MAPS", "false").lower() == "true"
        ),
        render_depth_maps=(
            os.environ.get("RENDER_DEPTH_MAPS", "false").lower() == "true"
        ),
        render_disparity_maps=(
            os.environ.get("RENDER_DISPARITY_MAPS", "false").lower() == "true"
        ),
    ),
    asset_files=UploadFiles(
        upload_files=[
            UploadFile(
                url="",
                file_upload=FileUpload(
                    file_details=FileDetails(
                        file_stem=file_path.stem, mime_type=file_path.suffix
                    ),
                    file_content=Base64StringContent(
                        base64_string_content=encode_file_to_base64(file_path=file_path)
                    ),
                ),
            )
            for file_path in asset_file_paths
        ]
    ),
)

asset_usda_file_path=PosixPath('/home/yunusskeete/Documents/code/github/spatialintelligence/eng-rendering-services/sample_objects/Accessories/Decor/SM_Decor01/SM_Decor01.usda')
usda_file_path=PosixPath('/home/yunusskeete/Documents/code/github/spatialintelligence/eng-rendering-services/sample_objects/Accessories/Decor/SM_Decor01/SM_Decor01.usda')
len(asset_file_paths)=5
asset_file_paths=[PosixPath('/home/yunusskeete/Documents/code/github/spatialintelligence/eng-rendering-services/sample_objects/Accessories/Decor/SM_Decor01/SM_Decor01.usda'), PosixPath('/home/yunusskeete/Documents/code/github/spatialintelligence/eng-rendering-services/sample_objects/Accessories/Decor/SM_Decor01/MI_Wood-Ash-PlainCut-Ebony.usda'), PosixPath('/home/yunusskeete/Documents/code/github/spatialintelligence/eng-rendering-services/sample_objects/Accessories/Decor/SM_Decor01/Textures/Game_Models_Model_assets_Materials_MI_Wood-Ash-PlainCut-Ebony_BaseColor.png'), PosixPath('/home/yunusskeete/Documents/code/github/spa

In [ ]:
render_asset_response_dict: Dict[str, Any] = (
    rendering_service_endpoint_config.post_json(
        json_payload=asset_rendering_request.model_dump(mode="json")
    )
)
render_asset_response = RenderAssetResponse(**render_asset_response_dict)

In [23]:
render_asset_response.uid

'abcde_12345'

In [24]:
render_asset_response.rendering_details

RenderingDetails(num_renders=6, only_northern_hemisphere=True, fast_mode=False, render_alpha_maps=False, render_depth_maps=False, render_disparity_maps=False)

In [25]:
render_asset_response.image_files

ImageFiles(images=[ImageFile(file_upload=FileUpload(file_stem=005, mime_type=.png, base64_string_content=iVBORw0KGgoAAAAN... (length=183036), , uid='abcde_12345', url=None), ImageFile(file_upload=FileUpload(file_stem=001, mime_type=.png, base64_string_content=iVBORw0KGgoAAAAN... (length=195704), , uid='abcde_12345', url=None), ImageFile(file_upload=FileUpload(file_stem=004, mime_type=.png, base64_string_content=iVBORw0KGgoAAAAN... (length=189312), , uid='abcde_12345', url=None), ImageFile(file_upload=FileUpload(file_stem=000, mime_type=.png, base64_string_content=iVBORw0KGgoAAAAN... (length=185724), , uid='abcde_12345', url=None), ImageFile(file_upload=FileUpload(file_stem=002, mime_type=.png, base64_string_content=iVBORw0KGgoAAAAN... (length=184168), , uid='abcde_12345', url=None), ImageFile(file_upload=FileUpload(file_stem=003, mime_type=.png, base64_string_content=iVBORw0KGgoAAAAN... (length=203216), , uid='abcde_12345', url=None)])